# 03 — Output and Visualization

**SOM Composites · Practically Perfect Hindcast (PPH) Analysis · Case Studies**

This notebook is the final stage of the NARR SOM pipeline.  It loads the trained SOM
and BMU assignments from notebook 02 and the PPH severe weather climatology dataset,
then produces publication-quality visualizations:

1. **Full SOM panel** — composite CAPE + CIN contours for every node on a CONUS map
2. **PPH frequency panel** — Practically Perfect Significant Severe per SOM node
3. **Best-node composite** — environment and PPH side-by-side for the highest-signal node
4. **Total composite** — all-date average environment and PPH frequency
5. **90th-percentile composite** — extreme-environment and extreme-PPH fields
6. **Case study** — single-date CAPE/CIN vs PPH for a specified high-impact event
7. **Node summary heatmaps** — PPH frequency, node counts, mean CAPE, mean CIN

### Input
- `cape_cin_NARR.zarr/` — merged NARR dataset (notebook 01)
- `som_output.pkl` — trained SOM and BMU assignments (notebook 02)
- `pper_all_sig_svr_1979_2023.nc` — Practically Perfect Hindcast dataset

### PPH Dataset
The Practically Perfect Hindcast (PPH) is a gridded probability field that represents
the observed climatological severe weather signal for any given day, constructed from
SPC storm reports smoothed with a Gaussian kernel.  Higher values indicate a greater
observed severe weather occurrence.

## 1. Imports

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pandas as pd
import pickle
import os

from scipy.interpolate import griddata
from scipy.ndimage import gaussian_filter

## 2. Configuration

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
ZARR_PATH  = "/path/to/output/cape_cin_NARR.zarr"      # notebook 01 output
PKL_PATH   = "/path/to/output/som_output.pkl"           # notebook 02 output
PPH_PATH   = "/path/to/pper_all_sig_svr_1979_2023.nc"  # PPH climatology
FIGURE_DIR = "/path/to/figures"                         # where to save plots

os.makedirs(FIGURE_DIR, exist_ok=True)

# ── Analysis parameters ───────────────────────────────────────────────────────
PPH_VAR    = "p_perfect_totsigsvr"   # variable name inside the PPH NetCDF
PPH_THRESH = 1.0                     # minimum PPH (%) to count as a "severe day"
SMOOTH_SIGMA = 1.2                   # Gaussian smoothing sigma for PPH contour plots

# ── PPH colour table (NWS/SPC style) ─────────────────────────────────────────
PPH_LEVELS = [0, 5, 15, 30, 45, 60, 75, 100]
PPH_COLORS = ["#ffffff", "#b07a3f", "#ffff00", "#ff0000",
              "#ff00ff", "#7a00ff", "#00d6d6"]
PPH_CMAP   = mcolors.ListedColormap(PPH_COLORS)
PPH_NORM   = mcolors.BoundaryNorm(PPH_LEVELS, PPH_CMAP.N)

print("Configuration set.")

## 3. Load Data

Load the NARR environment dataset, the trained SOM + BMU assignments, and the PPH
climatology.  All three are required before any analysis can proceed.

In [ ]:
# ── NARR environment ──────────────────────────────────────────────────────────
ds = xr.open_zarr(ZARR_PATH)
print("NARR dataset loaded:", dict(ds.sizes))

# ── Trained SOM and BMUs ──────────────────────────────────────────────────────
with open(PKL_PATH, "rb") as f:
    som_data = pickle.load(f)

som   = som_data["som"]
bmus  = som_data["bmus"]
SOM_X = som_data["som_x"]
SOM_Y = som_data["som_y"]

print(f"SOM loaded: {SOM_X}×{SOM_Y} grid, {len(bmus)} BMUs")

# ── PPH climatology ───────────────────────────────────────────────────────────
pph = xr.open_dataset(PPH_PATH)
print("\nPPH dataset:")
print(pph)

## 4. Align NARR and PPH Date Ranges

The NARR and PPH datasets may cover different date ranges.  We normalise both time
axes to midnight (dropping sub-daily offsets), find the common dates, and subset both
datasets so that each row index in `bmus_matched` corresponds to the same calendar
day in `ds_matched` and `pph_matched`.

In [ ]:
# Normalise to date-only (midnight)
ds_dates  = pd.to_datetime(ds["time"].values).normalize()
pph_dates = pd.to_datetime(pph["time"].values).normalize()

# Attach as coordinate for easy .sortby()
ds  = ds.assign_coords(date=("time", ds_dates))
pph = pph.assign_coords(date=("time", pph_dates))

# Intersection of available dates
common_dates = np.intersect1d(ds_dates, pph_dates)
print(f"NARR dates   : {len(ds_dates)}")
print(f"PPH dates    : {len(pph_dates)}")
print(f"Common dates : {len(common_dates)}")

# Subset to common dates
ds_keep  = np.isin(ds_dates, common_dates)
pph_keep = np.isin(pph_dates, common_dates)

ds_matched  = ds.isel(time=ds_keep)
pph_matched = pph.isel(time=pph_keep)
bmus_matched = bmus[ds_keep]

# Sort both datasets by date to guarantee row alignment
ds_matched  = ds_matched.sortby("date")
pph_matched = pph_matched.sortby("date")
sort_idx    = np.argsort(ds_dates[ds_keep])
bmus_matched = bmus_matched[sort_idx]

# Confirm alignment
assert np.all(ds_matched["date"].values == pph_matched["date"].values), \
    "Date arrays do not match after sorting!"

print("\nDate alignment verified.")
print(f"Matched samples: {len(bmus_matched)}")

## 5. Build Interpolation Grid

The PPH dataset is on a regular lat/lon grid while NARR uses Lambert Conformal
projection coordinates.  We precompute the source and target grids here once so
they can be reused across all subsequent plots.

In [ ]:
# Determine lat/lon coordinate names in the PPH dataset
lat_name = "lat" if "lat" in pph_matched else "latitude"
lon_name = "lon" if "lon" in pph_matched else "longitude"

# PPH source grid — flatten to (n_points, 2)
source_lon = pph_matched[lon_name].values
source_lat = pph_matched[lat_name].values
source_points = np.column_stack([source_lon.ravel(), source_lat.ravel()])

# NARR target grid
target_lon = ds["lon"].values
target_lat = ds["lat"].values

print(f"PPH source grid : {source_lon.shape}  →  {source_points.shape[0]} points")
print(f"NARR target grid: {target_lon.shape}")


def interpolate_pph_to_narr(pph_field):
    """Interpolate a 2-D PPH field onto the NARR grid with edge-fill and smoothing."""
    linear  = griddata(source_points, pph_field.ravel(), (target_lon, target_lat), method="linear")
    nearest = griddata(source_points, pph_field.ravel(), (target_lon, target_lat), method="nearest")
    interp  = np.where(np.isnan(linear), nearest, linear)
    return gaussian_filter(interp, sigma=SMOOTH_SIGMA)

## 6. Full SOM Environmental Composite Panel

Each panel shows the composite mean CAPE (colour fill) and CIN (hatching where
CIN ≤ −50 J/kg) for the days that mapped to that SOM node.  This is the primary
diagnostic for understanding what synoptic patterns the SOM has identified.

The plot is saved to `FIGURE_DIR/SOM_Out.png`.

In [ ]:
fig, axes = plt.subplots(
    SOM_Y, SOM_X,
    figsize=(18, 13),
    subplot_kw={"projection": ccrs.PlateCarree()},
    constrained_layout=True
)
fig.suptitle("Environmental Feature Input SOM — Mean CAPE and CIN by Node", fontsize=24)

for j in range(SOM_Y):
    for i in range(SOM_X):
        ax = axes[j, i]
        node_mask = (bmus[:, 0] == i) & (bmus[:, 1] == j)
        n_cases   = node_mask.sum()

        if n_cases == 0:
            ax.set_title(f"Node {j+1},{i+1}\nNo cases")
            ax.axis("off")
            continue

        cape_node = ds["cape"].isel(time=node_mask).mean("time")
        cin_node  = ds["cin"].isel(time=node_mask).mean("time")

        pm = ax.pcolormesh(
            ds["lon"], ds["lat"], cape_node,
            transform=ccrs.PlateCarree(), shading="auto", cmap="plasma"
        )
        ax.contourf(
            ds["lon"], ds["lat"], cin_node,
            levels=[-1e9, -50], colors="none", hatches=["/////"],
            transform=ccrs.PlateCarree()
        )

        ax.set_extent([-125, -66, 24, 50], crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
        ax.add_feature(cfeature.STATES, linewidth=0.3)
        ax.add_feature(cfeature.BORDERS, linewidth=0.5)
        ax.set_title(f"Node {j+1},{i+1}  |  n={n_cases}")

fig.colorbar(pm, ax=axes.ravel().tolist(), shrink=0.8,
             orientation="horizontal", label="Mean CAPE (J/kg)")

legend_handles = [
    mpatches.Patch(facecolor="none", edgecolor="black", hatch="////", label="CIN ≤ −50 J/kg")
]
fig.legend(handles=legend_handles, loc="lower center",
           bbox_to_anchor=(0.5, -0.02), ncol=1, fontsize=12)

out_path = os.path.join(FIGURE_DIR, "SOM_Out.png")
plt.savefig(out_path, dpi=300, bbox_inches="tight")
print(f"Saved: {out_path}")
plt.show()

## 7. PPH Frequency Panel by SOM Node

For each node we compute the fraction of node days on which the PPH exceeds
`PPH_THRESH` at any grid point, then interpolate and smooth that field onto the NARR
grid for display.  Higher values indicate that days assigned to that node had
elevated observed severe weather activity.

The PPH colour table mirrors the SPC practically perfect format:
- **Brown** 5–15 % | **Yellow** 15–30 % | **Red** 30–45 %
- **Magenta** 45–60 % | **Purple** 60–75 % | **Cyan** > 75 %

In [ ]:
fig, axes = plt.subplots(
    SOM_Y, SOM_X,
    figsize=(18, 13),
    subplot_kw={"projection": ccrs.PlateCarree()},
    constrained_layout=True
)
fig.suptitle("Practically Perfect Significant Severe — Frequency by SOM Node", fontsize=24)

for j in range(SOM_Y):
    for i in range(SOM_X):
        ax = axes[j, i]
        node_mask = (bmus_matched[:, 0] == i) & (bmus_matched[:, 1] == j)
        n_cases   = node_mask.sum()

        if n_cases == 0:
            ax.set_title(f"Node {j+1},{i+1}\nNo cases")
            ax.axis("off")
            continue

        pph_node = (
            (pph_matched[PPH_VAR].isel(time=node_mask) >= PPH_THRESH)
            .mean("time") * 100.0
        )

        pph_smooth = interpolate_pph_to_narr(pph_node.values)

        pm = ax.contourf(
            target_lon, target_lat, pph_smooth,
            levels=PPH_LEVELS, cmap=PPH_CMAP, norm=PPH_NORM, extend="max",
            transform=ccrs.PlateCarree()
        )

        ax.set_extent([-125, -66, 24, 50], crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
        ax.add_feature(cfeature.STATES, linewidth=0.3)
        ax.add_feature(cfeature.BORDERS, linewidth=0.5)
        ax.set_title(f"Node {j+1},{i+1}  |  n={n_cases}")

fig.colorbar(pm, ax=axes.ravel().tolist(), shrink=0.8,
             orientation="horizontal", pad=0.03,
             ticks=[5, 15, 30, 45, 60, 75],
             label=f"% of Node Days with PPH ≥ {PPH_THRESH}%")

out_path = os.path.join(FIGURE_DIR, "PPH_Out.png")
plt.savefig(out_path, dpi=300, bbox_inches="tight")
print(f"Saved: {out_path}")
plt.show()

## 8. Best-Performing SOM Node

The *best-performing* node is defined as the one with the highest maximum PPH
frequency signal across its spatial domain — i.e., the node whose composite days
are most strongly associated with significant severe weather.

We display the environmental composite and PPH frequency side-by-side for this node.

In [ ]:
# ── Find best node ────────────────────────────────────────────────────────────
node_score = np.full((SOM_Y, SOM_X), np.nan)
node_n     = np.zeros((SOM_Y, SOM_X), dtype=int)

for j in range(SOM_Y):
    for i in range(SOM_X):
        mask = (bmus_matched[:, 0] == i) & (bmus_matched[:, 1] == j)
        node_n[j, i] = mask.sum()
        if mask.sum() == 0:
            continue
        pph_freq = (
            (pph_matched[PPH_VAR].isel(time=mask) >= PPH_THRESH)
            .mean("time") * 100.0
        )
        node_score[j, i] = float(pph_freq.max())

best_j, best_i = np.unravel_index(np.nanargmax(node_score), node_score.shape)

print(f"Best node : row {best_j+1}, col {best_i+1}")
print(f"n cases   : {node_n[best_j, best_i]}")
print(f"PPH score : {node_score[best_j, best_i]:.2f} %")

# ── Build composites ──────────────────────────────────────────────────────────
node_mask = (bmus_matched[:, 0] == best_i) & (bmus_matched[:, 1] == best_j)
n_cases   = node_mask.sum()

cape_node = ds["cape"].isel(time=node_mask).mean("time")
cin_node  = ds["cin"].isel(time=node_mask).mean("time")

pph_node = (
    (pph_matched[PPH_VAR].isel(time=node_mask) >= PPH_THRESH)
    .mean("time") * 100.0
)
pph_smooth = interpolate_pph_to_narr(pph_node.values)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(
    1, 2, figsize=(15, 6),
    subplot_kw={"projection": ccrs.PlateCarree()},
    constrained_layout=True
)
fig.suptitle(f"Best SOM Node ({best_j+1},{best_i+1})  |  n = {n_cases}",
             fontsize=20, y=1.02)

# Left: environmental composite
ax = axes[0]
cape_pm = ax.pcolormesh(ds["lon"], ds["lat"], cape_node,
                        cmap="plasma", shading="auto", transform=ccrs.PlateCarree())
ax.contourf(ds["lon"], ds["lat"], cin_node,
            levels=[-1e9, -50], colors="none", hatches=["/////"],
            transform=ccrs.PlateCarree())
ax.set_extent([-125, -66, 24, 50])
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.STATES, linewidth=0.3)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.set_title("Environmental Composite — Mean CAPE + CIN")
fig.colorbar(cape_pm, ax=ax, orientation="horizontal", pad=0.04).set_label("Mean CAPE (J/kg)")

# Right: PPH frequency
ax = axes[1]
pm = ax.contourf(target_lon, target_lat, pph_smooth,
                 levels=PPH_LEVELS, cmap=PPH_CMAP, norm=PPH_NORM,
                 extend="max", transform=ccrs.PlateCarree())
ax.set_extent([-125, -66, 24, 50])
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.STATES, linewidth=0.3)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.set_title(f"PPH Frequency  (≥ {PPH_THRESH} %)")
fig.colorbar(pm, ax=ax, orientation="horizontal", pad=0.04,
             ticks=[5, 15, 30, 45, 60, 75]).set_label("% of Node Days")

out_path = os.path.join(FIGURE_DIR, "Best_Node.png")
plt.savefig(out_path, dpi=300, bbox_inches="tight")
print(f"Saved: {out_path}")
plt.show()

## 9. Total Composite — All Dates

The total composite averages over all matched dates regardless of SOM node assignment.
This provides a climatological baseline against which individual node composites can
be compared.

In [ ]:
cape_total = ds["cape"].mean("time")
cin_total  = ds["cin"].mean("time")

pph_total = (
    (pph_matched[PPH_VAR] >= PPH_THRESH)
    .mean("time") * 100.0
)
pph_total_smooth = interpolate_pph_to_narr(pph_total.values)

fig, axes = plt.subplots(
    1, 2, figsize=(15, 6),
    subplot_kw={"projection": ccrs.PlateCarree()},
    constrained_layout=True
)
fig.suptitle("Total Composite: Mean Environment and PPH Frequency (All Dates)",
             fontsize=16, y=1.02)

ax = axes[0]
cape_pm = ax.pcolormesh(ds["lon"], ds["lat"], cape_total,
                        cmap="plasma", shading="auto", transform=ccrs.PlateCarree())
ax.contourf(ds["lon"], ds["lat"], cin_total,
            levels=[-1e9, -50], colors="none", hatches=["/////"],
            transform=ccrs.PlateCarree())
ax.set_extent([-125, -66, 24, 50], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.STATES, linewidth=0.3)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.set_title("Mean CAPE + CIN (All Dates)")
fig.colorbar(cape_pm, ax=ax, orientation="horizontal",
             pad=0.04, shrink=0.85).set_label("CAPE (J/kg)")

ax = axes[1]
pph_pm = ax.contourf(target_lon, target_lat, pph_total_smooth,
                     levels=PPH_LEVELS, cmap=PPH_CMAP, norm=PPH_NORM,
                     extend="max", transform=ccrs.PlateCarree())
ax.set_extent([-125, -66, 24, 50], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.STATES, linewidth=0.3)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.set_title(f"PPH Frequency (All Dates, ≥ {PPH_THRESH} %)")
fig.colorbar(pph_pm, ax=ax, orientation="horizontal", pad=0.04,
             shrink=0.85, ticks=[5, 15, 30, 45, 60, 75]).set_label("% of All Dates")

out_path = os.path.join(FIGURE_DIR, "Total_Composite.png")
plt.savefig(out_path, dpi=300, bbox_inches="tight")
print(f"Saved: {out_path}")
plt.show()

## 10. 90th-Percentile Composite

Instead of the mean, we show the 90th percentile of CAPE (extreme instability) and
the 10th percentile of CIN (extreme inhibition — most negative) to highlight the
tail of the distribution, alongside the 90th percentile of raw PPH values.

> The dataset is re-chunked to a single time chunk before calling `.quantile()` because
> Dask requires all data for a reduction to be in the same chunk.

In [ ]:
# Rechunk for quantile computation
ds_q  = ds_matched.chunk({"time": -1})
pph_q = pph_matched.chunk({"time": -1})

cape_p90 = ds_q["cape"].quantile(0.90, dim="time").squeeze(drop=True)
cin_p10  = ds_q["cin"].quantile(0.10, dim="time").squeeze(drop=True)   # most negative CIN
pph_p90  = pph_q[PPH_VAR].quantile(0.90, dim="time").squeeze(drop=True)

pph_p90_smooth = interpolate_pph_to_narr(pph_p90.values)

fig, axes = plt.subplots(
    1, 2, figsize=(15, 6),
    subplot_kw={"projection": ccrs.PlateCarree()},
    constrained_layout=True
)
fig.suptitle("90th-Percentile Composite: Extreme Environment and PPH",
             fontsize=16, y=1.02)

ax = axes[0]
cape_pm = ax.pcolormesh(ds_matched["lon"], ds_matched["lat"], cape_p90,
                        cmap="plasma", shading="auto", transform=ccrs.PlateCarree())
ax.contourf(ds_matched["lon"], ds_matched["lat"], cin_p10,
            levels=[-1e9, -50], colors="none", hatches=["/////"],
            transform=ccrs.PlateCarree())
ax.set_extent([-125, -66, 24, 50])
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.STATES, linewidth=0.3)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.set_title("CAPE (90th pct) + CIN (10th pct)")
fig.colorbar(cape_pm, ax=ax, orientation="horizontal",
             pad=0.04, shrink=0.85).set_label("CAPE (J/kg)")

ax = axes[1]
pph_pm = ax.contourf(target_lon, target_lat, pph_p90_smooth,
                     levels=PPH_LEVELS, cmap=PPH_CMAP, norm=PPH_NORM,
                     extend="max", transform=ccrs.PlateCarree())
ax.set_extent([-125, -66, 24, 50])
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.STATES, linewidth=0.3)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.set_title("PPH (90th percentile)")
fig.colorbar(pph_pm, ax=ax, orientation="horizontal", pad=0.04, shrink=0.85,
             ticks=[5, 15, 30, 45, 60, 75]).set_label("Practically Perfect Sig Severe (%)")

out_path = os.path.join(FIGURE_DIR, "P90_Composite.png")
plt.savefig(out_path, dpi=300, bbox_inches="tight")
print(f"Saved: {out_path}")
plt.show()

## 11. Case Study — Single High-Impact Date

This section examines a specific high-impact event date.  The cell finds:
1. The date with the highest single-day PPH maximum in the record
2. The CAPE/CIN environment on a user-specified `TARGET_DATE`
3. Which SOM node was assigned to `TARGET_DATE` and its node composite

Change `TARGET_DATE` to any date of interest within the dataset range.

In [ ]:
TARGET_DATE = pd.Timestamp("2020-04-12")   # ← Change to the date you want to examine

In [ ]:
# Find the date with the highest domain-maximum PPH in the record
pph_daymax = pph_matched[PPH_VAR].max(dim=("y", "x"))
best_t     = int(pph_daymax.argmax().values)
best_date  = pd.to_datetime(pph_matched["date"].isel(time=best_t).values)

print(f"Maximum PPH date  : {best_date:%Y-%m-%d}")
print(f"Maximum PPH value : {float(pph_daymax.isel(time=best_t).values):.1f} %")

# Find environment index for TARGET_DATE
env_dates = pd.to_datetime(ds_matched["date"].values).normalize()
env_idx   = np.where(env_dates == TARGET_DATE)[0]

if len(env_idx) == 0:
    raise ValueError(f"{TARGET_DATE:%Y-%m-%d} not found in matched dataset.")
env_idx = env_idx[0]

cape_single = ds_matched["cape"].isel(time=env_idx)
cin_single  = ds_matched["cin"].isel(time=env_idx)
pph_single  = pph_matched[PPH_VAR].isel(time=best_t)

pph_single_smooth = interpolate_pph_to_narr(pph_single.values)

# SOM node for TARGET_DATE
target_i, target_j = bmus_matched[env_idx]
print(f"\n{TARGET_DATE:%Y-%m-%d} → SOM Node row {target_j+1}, col {target_i+1}")

# Plot
fig, axes = plt.subplots(
    1, 2, figsize=(15, 6),
    subplot_kw={"projection": ccrs.PlateCarree()},
    constrained_layout=True
)
fig.suptitle(
    f"Case Study: Max-PPH Date ({best_date:%Y-%m-%d}) and CAPE/CIN on {TARGET_DATE:%Y-%m-%d}",
    fontsize=14, y=1.03
)

ax = axes[0]
pph_pm = ax.contourf(target_lon, target_lat, pph_single_smooth,
                     levels=PPH_LEVELS, cmap=PPH_CMAP, norm=PPH_NORM,
                     extend="max", transform=ccrs.PlateCarree())
ax.set_extent([-125, -66, 24, 50], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.STATES, linewidth=0.3)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.set_title(f"Max Single-Date PPH  ({best_date:%Y-%m-%d})")
fig.colorbar(pph_pm, ax=ax, orientation="horizontal", pad=0.04, shrink=0.85,
             ticks=[5, 15, 30, 45, 60, 75]).set_label("Practically Perfect Sig Severe (%)")

ax = axes[1]
cape_pm = ax.pcolormesh(ds_matched["lon"], ds_matched["lat"], cape_single,
                        cmap="plasma", shading="auto", transform=ccrs.PlateCarree())
ax.contourf(ds_matched["lon"], ds_matched["lat"], cin_single,
            levels=[-1e9, -50], colors="none", hatches=["/////"],
            transform=ccrs.PlateCarree())
ax.set_extent([-125, -66, 24, 50], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.STATES, linewidth=0.3)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.set_title(f"CAPE + CIN  ({TARGET_DATE:%Y-%m-%d})\nSOM Node {target_j+1},{target_i+1}")
fig.colorbar(cape_pm, ax=ax, orientation="horizontal",
             pad=0.04, shrink=0.85).set_label("CAPE (J/kg)")

out_path = os.path.join(FIGURE_DIR, "Case_Study.png")
plt.savefig(out_path, dpi=300, bbox_inches="tight")
print(f"Saved: {out_path}")
plt.show()

## 12. Summary Heatmap — PPH Frequency by Node

A compact single-panel view showing the fraction of days in each SOM node that
contained a domain-maximum PPH value above `PPH_THRESH`.  This quickly identifies
which nodes are associated with the most severe weather activity.

In [ ]:
pph_daymax_vals = pph_matched[PPH_VAR].max(dim=("y", "x")).values

node_pph = np.full((SOM_Y, SOM_X), np.nan)

for j in range(SOM_Y):
    for i in range(SOM_X):
        mask = (bmus_matched[:, 0] == i) & (bmus_matched[:, 1] == j)
        if mask.sum() > 0:
            node_pph[j, i] = np.mean(pph_daymax_vals[mask] >= PPH_THRESH) * 100

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(node_pph, cmap="viridis", vmin=0, vmax=100)

for j in range(SOM_Y):
    for i in range(SOM_X):
        if not np.isnan(node_pph[j, i]):
            ax.text(i, j, f"{node_pph[j,i]:.1f}", ha="center", va="center",
                    color="white", fontsize=10, fontweight="bold")

ax.set_title(f"PPH Frequency by SOM Node  (PPH ≥ {PPH_THRESH} %, %)")
ax.set_xlabel("SOM Column")
ax.set_ylabel("SOM Row")
plt.colorbar(im, ax=ax, label="% of Node Days with Severe")
plt.tight_layout()

out_path = os.path.join(FIGURE_DIR, "PPH_Heatmap.png")
plt.savefig(out_path, dpi=300, bbox_inches="tight")
print(f"Saved: {out_path}")
plt.show()

---
## Summary of Saved Figures

| File | Description |
|---|---|
| `SOM_Out.png` | Full 5×5 CAPE/CIN composite panel for all SOM nodes |
| `PPH_Out.png` | Full 5×5 PPH frequency panel for all SOM nodes |
| `Best_Node.png` | Environment + PPH composite for the highest-signal node |
| `Total_Composite.png` | Climatological mean environment and PPH frequency |
| `P90_Composite.png` | 90th-percentile environment and PPH |
| `Case_Study.png` | Single-date CAPE/CIN vs max-PPH event |
| `PPH_Heatmap.png` | Compact node-level PPH frequency summary |